# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [2]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 5000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test


5000 test impressions, observed CTR 0.0358


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_cb81877a547572a6,2026-11-14 02:57:17.000000000,u_71239e2d,"{'device': 'android', 'page': 'category', 'reg...","[{'product_id': 'p_0dde7e', 'ts': '2026-11-14 ...","[p_69d40e, p_2156a6, p_cbcc44, p_ada8ad, p_8a8...","[0.451662, 0.00903, 0.140389, 0.09037, 0.13621...",NaN,NaN
1,r_c1b9c64a98f96687,2026-11-11 20:21:25.337000000,u_83c88ccb,"{'device': 'web', 'page': 'home', 'region': 'C...",[],"[p_adc3e6, p_09a938, p_68498e, p_050730, p_c46...","[0.268636, 0.154529, 0.012855, 0.067831, 0.053...",NaN,NaN
2,r_fba9e6f9f80d6f5e,2026-11-09 16:35:45.508000000,u_b222276e,"{'device': 'web', 'page': 'home', 'region': 'C...",[],"[p_f039e8, p_8a86a8, p_c72f4f, p_ba0231, p_2d7...","[0.005778, 0.058213, 0.011492, 0.01693, 0.0093...",NaN,NaN


In [3]:
covisit = {}
print("coview omitted in the quick baseline/Hercules comparison")

coview omitted in the quick baseline/Hercules comparison


In [4]:
# c04: importar nuestro recommender con artefactos de corte
import os, sys
os.environ["MODEL_SUFFIX"] = "_cut"
sys.path.insert(0, ".")
sys.modules.pop("recommender", None)     # evita la versión cacheada
import recommender10 as ours

assert ours.LGB8_MODEL is not None, "LGBM8 no se carg?"

# Ranking de los más vendidos ANTES del corte (el baseline real, sin ver el futuro)
best = sql(f"""
    SELECT product_id::String AS product_id, count() AS n
    FROM shop.history_events
    WHERE event_type = 'purchase' AND ts::DateTime64(3) < toDateTime('{CUT}')
    GROUP BY product_id ORDER BY n DESC LIMIT 200""").product_id.tolist()

Cargando modelo LightGBM (17 features) desde C:\Users\pablo\Videos\Cosas de Pablo\Datathon\TinyBird\starter\models\modelo_lgbm7.pkl...


C:\Users\pablo\Videos\Cosas de Pablo\Datathon\TinyBird\starter\venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Ã‚Â¡Modelo de 17 features cargado correctamente!


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [5]:
# c07: políticas
def baseline(row):
    """Los 10 más vendidos antes del corte (sin filtro de stock offline)."""
    return best[:10]

def coview(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views or [])):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

def hercules(row):
    request_ts = ours._parse_ts(row.ts)
    views = row.recent_views if isinstance(row.recent_views, list) else []
    if row.user_id is not None and request_ts is not None:
        for event in views:
            if not isinstance(event, dict):
                continue
            event_ts = ours._parse_ts(event.get("ts"))
            if event_ts is not None and event_ts < request_ts:
                ours._record_user_event({"user_id": row.user_id, "product_id": event.get("product_id"),
                                         "event_type": "view", "ts": event_ts})
    return ours.recommend(views, best, user_id=row.user_id, request_ts=request_ts,
                          is_new_user=getattr(row, "is_new_user", None))

POLICIES = {"baseline": baseline, "Hercules": hercules}

## Evaluation

In [6]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    reciprocal_rank = [1 / (r.index(c) + 1) if c in r else 0.0
                       for c, r, has in zip(clicked, recs, has_click) if has]
    return {"hit_rate@10": hit, "MRR@10": float(np.mean(reciprocal_rank)),
            "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0112


,hit_rate@10,MRR@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.307263,0.15589,0.005956,0.041667,192.0
Hercules,0.307263,0.15589,0.005956,0.041667,192.0


In [7]:
rng = np.random.default_rng(0)
B = 2000
N = len(test)

def per_row(policy, df):
    """Resultados por fila de una política (una sola pasada por el test)."""
    recs = [policy(r) for r in df.itertuples()]
    shown1 = df.shown.str[0].to_numpy()
    clicked = df.clicked.to_numpy()
    prop1 = df.propensity.str[0].to_numpy().astype(float)
    match = np.array([r[0] == s for r, s in zip(recs, shown1)], dtype=float)
    reward = np.array([c == s for c, s in zip(clicked, shown1)], dtype=float)
    w = match / prop1
    hit = np.array([np.nan if pd.isna(c) else float(c in r)
                    for c, r in zip(clicked, recs)])
    mrr = np.array([np.nan if pd.isna(c) else (1 / (r.index(c) + 1) if c in r else 0.0)
                    for c, r in zip(clicked, recs)])
    return {"num": w * reward, "w": w, "hit": hit, "mrr": mrr}

R = {name: per_row(p, test) for name, p in POLICIES.items()}

def stats(res, idx):
    """IPS, SNIPS y hit@10 sobre un conjunto de filas (índices)."""
    num, w, hit, mrr = res["num"][idx], res["w"][idx], res["hit"][idx], res["mrr"][idx]
    return np.array([num.mean(),
                     num.sum() / max(w.sum(), 1e-9),
                     np.nanmean(hit),
                     np.nanmean(mrr)])

# Los mismos índices bootstrap para todas las políticas: así las diferencias son pareadas
IDX = [rng.integers(0, N, N) for _ in range(B)]
ALL = np.arange(N)
names = ["IPS slot1", "SNIPS slot1", "hit@10", "MRR@10"]

def summary(name):
    point = stats(R[name], ALL)
    boot = np.array([stats(R[name], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"valor": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

for name in POLICIES:
    print(f"\n== {name} ==")
    print(summary(name).round(4))

def diff(a, b):
    """Diferencia pareada a − b con IC95 (si el IC no incluye 0, la diferencia es significativa)."""
    point = stats(R[a], ALL) - stats(R[b], ALL)
    boot = np.array([stats(R[a], i) - stats(R[b], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"diff": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

print("\n== Hercules − baseline ==")
print(diff("Hercules", "baseline").round(4))


== baseline ==


              valor  IC95_inf  IC95_sup
IPS slot1    0.0060    0.0022    0.0104
SNIPS slot1  0.0417    0.0156    0.0708
hit@10       0.3073    0.2440    0.3787
MRR@10       0.1559    0.1128    0.2020

== Hercules ==


              valor  IC95_inf  IC95_sup
IPS slot1    0.0060    0.0022    0.0104
SNIPS slot1  0.0417    0.0156    0.0708
hit@10       0.3073    0.2440    0.3787
MRR@10       0.1559    0.1128    0.2020

== Hercules − baseline ==


             diff  IC95_inf  IC95_sup
IPS slot1     0.0       0.0       0.0
SNIPS slot1   0.0       0.0       0.0
hit@10        0.0       0.0       0.0
MRR@10        0.0       0.0       0.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.